# 🌸 KNN Applied on Iris Dataset
### Atlas Skilltech University — Machine Learning Course
---
**Is notebook mein hum sikhenge:**
- Real dataset par KNN kaise apply karte hain
- Complete pipeline: Data Loading → EDA → Preprocessing → Model Training → Evaluation
- Iris dataset — ek classic classification problem
- KNN Capstone Project

**Iris Dataset kya hai?**
> Iris ek flower (phool) ka naam hai. Is dataset mein 3 types ke flowers hain:
> - **Setosa** (Type 0)
> - **Versicolor** (Type 1)  
> - **Virginica** (Type 2)
>
> Har flower ke 4 measurements hain: Sepal Length, Sepal Width, Petal Length, Petal Width
> **Goal:** In measurements se predict karo — kaunsa flower kaunsa type hai?

## Step 1: Libraries Import Karo
---
**Keyword Explanation:**
- **seaborn** — ek library jo built-in datasets deti hai, aur sundar graphs banati hai
- **LabelEncoder** — text labels (Setosa/Versicolor/Virginica) ko numbers mein convert karta hai
- **KNeighborsClassifier** — sklearn ka KNN model

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report, accuracy_score
)

import warnings
warnings.filterwarnings('ignore')

print("✅ Saari libraries successfully import ho gayi!")

## Step 2: Data Loading — Dataset Load Karo
---
**Keyword Explanation:**
- **sns.load_dataset('iris')** — seaborn ka built-in Iris dataset load karta hai
- **shape** — dataframe mein kitni rows aur columns hain
- **dtypes** — har column ka data type (int, float, object)

In [ ]:
# Iris dataset load karo
df = sns.load_dataset('iris')

print("Dataset ka shape:", df.shape)
print("\nPehli 5 rows:")
df.head()

In [ ]:
print("Dataset ki basic info:")
print(df.info())

print("\nHar species kitni baar hai:")
print(df['species'].value_counts())

In [ ]:
print("Statistical Summary:")
df.describe().round(2)

## Step 3: EDA — Exploratory Data Analysis
---
**EDA kya hota hai?**
> Data ko achhi tarah samajhne ka process. Graphs aur statistics se dekho ki data kaisa hai, koi pattern hai ya nahin.

**Hum kya dekhenge:**
1. Missing values check
2. Species distribution
3. Feature distributions
4. Pairplot — features ke beech relationship
5. Correlation heatmap
6. Boxplot — feature spread by species

In [ ]:
# Missing values check karo
print("Missing Values (Khali jagahein):")
print(df.isnull().sum())
print("\n✅ Koi missing value nahi! Clean dataset hai.")

In [ ]:
# Species distribution — pie chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
species_count = df['species'].value_counts()
axes[0].bar(species_count.index, species_count.values, 
            color=['#FF6B6B', '#4ECDC4', '#45B7D1'], edgecolor='black')
axes[0].set_title('Har Species ki Count', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Species (Phool ka Prakar)')
axes[0].set_ylabel('Count (Ginati)')

# Pie chart
axes[1].pie(species_count.values, labels=species_count.index,
            autopct='%1.1f%%', colors=['#FF6B6B', '#4ECDC4', '#45B7D1'],
            startangle=90)
axes[1].set_title('Species Distribution (Percentage)', fontsize=13, fontweight='bold')

plt.suptitle('Iris Dataset — Species Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print("Observation: Teen species hain — har ek mein exactly 50 samples!")

In [ ]:
# Feature distributions — histograms
features = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, (feat, color) in enumerate(zip(features, colors)):
    axes[i].hist(df[feat], bins=20, color=color, edgecolor='black', alpha=0.8)
    axes[i].set_title(f'{feat} ka Distribution', fontsize=11)
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('Frequency (Baar)')
    axes[i].axvline(df[feat].mean(), color='red', linestyle='--', 
                    label=f'Mean: {df[feat].mean():.2f}')
    axes[i].legend()

plt.suptitle('Charo Features ki Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Pairplot — features ke beech relationship
# Yeh sabse important plot hai KNN ke liye!
print("Pairplot bana rahe hain — thoda waqt lagega...")

sns.pairplot(df, hue='species', 
             palette={'setosa': '#FF6B6B', 'versicolor': '#4ECDC4', 'virginica': '#45B7D1'},
             diag_kind='hist', plot_kws={'alpha': 0.7})

plt.suptitle('Pairplot — Saare Features Ek Saath Dekho', 
             fontsize=14, fontweight='bold', y=1.02)
plt.show()
print("Observation: Petal features (length aur width) species ko clearly separate karte hain!")

In [ ]:
# Boxplot — species ke according feature spread
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, feat in enumerate(features):
    df.boxplot(column=feat, by='species', ax=axes[i],
               boxprops=dict(color='navy'),
               medianprops=dict(color='red', linewidth=2))
    axes[i].set_title(f'{feat} by Species', fontsize=11)
    axes[i].set_xlabel('Species')
    axes[i].set_ylabel(feat)

plt.suptitle('Boxplot — Species ke Hisaab se Features', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation Heatmap
numeric_df = df.select_dtypes(include=[np.number])

plt.figure(figsize=(8, 6))
sns.heatmap(numeric_df.corr(), annot=True, fmt='.2f', 
            cmap='RdYlGn', center=0,
            linewidths=0.5, linecolor='white',
            annot_kws={'size': 12})
plt.title('Correlation Heatmap — Features ke Beech Sambandh', 
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print("Observation: Petal length aur petal width mein bahut zyada correlation (0.96) hai!")

## Step 4: Preprocessing — Data Taiyaar Karo
---
**Keyword Explanation:**
- **LabelEncoder** — text labels ko numbers mein convert karta hai
  - setosa → 0, versicolor → 1, virginica → 2
- **train_test_split** — data ko train aur test mein baantna
- **stratify** — ensure karta hai ki train aur test dono mein species equally divided ho
- **StandardScaler** — features ko same scale par laana (KNN ke liye MUST hai!)

In [ ]:
# Label Encoding — species text ko numbers mein
le = LabelEncoder()
df['species_encoded'] = le.fit_transform(df['species'])

print("Label Encoding kya hua:")
encoding_map = dict(zip(le.classes_, le.transform(le.classes_)))
for name, num in encoding_map.items():
    print(f"  {name} → {num}")

print("\nDataframe abhi kaisa dikhta hai:")
df[['species', 'species_encoded']].head(10)

In [ ]:
# Features (X) aur Target (y) alag karo
X = df[features].values  # Charon measurements
y = df['species_encoded'].values  # Species (0, 1, 2)

print("Features (X) ka shape:", X.shape)
print("Target (y) ka shape:", y.shape)
print("\nX ke pehle 3 samples:")
print(X[:3])
print("\ny ke pehle 3 values:", y[:3])

In [ ]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% test data
    random_state=42,    # Reproducibility ke liye
    stratify=y          # Har species equally divide ho
)

print("Data baant diya gaya:")
print(f"  Training data:  {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"  Testing data:   {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.0f}%)")

print("\nTraining mein species distribution:")
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  Class {u} ({le.inverse_transform([u])[0]}): {c} samples")

In [ ]:
# Feature Scaling — StandardScaler
# KNN ke liye scaling ZARURI hai!
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # Train par fit karo
X_test_scaled = scaler.transform(X_test)         # Test par sirf transform

print("Scaling se pehle (Training data ka mean):")
for i, feat in enumerate(features):
    print(f"  {feat}: Mean = {X_train[:, i].mean():.2f}, Std = {X_train[:, i].std():.2f}")

print("\nScaling ke baad (Training data ka mean):")
for i, feat in enumerate(features):
    print(f"  {feat}: Mean = {X_train_scaled[:, i].mean():.4f}, Std = {X_train_scaled[:, i].std():.4f}")

print("\n✅ Ab sabhi features ek hi scale par hain (mean≈0, std≈1)")

## Step 5: Best K Dhundo — Elbow Method
---
**Elbow Method kya hai?**
> Alag-alag K values try karo aur dekho kaunsi K value par accuracy sabse zyada hai.
> Jahan accuracy stable ho jaaye (elbow jaisa shape) — wahi best K hai.

**Keyword:**
- **Overfitting** — K=1 par model training data ko ratta maar leta hai, naya data fail hoga
- **Underfitting** — K bahut bada ho to model kuch bhi nahi seekhta
- **Sweet Spot** — beech wali K value best hoti hai

In [ ]:
# Elbow Method — Best K dhundo
k_values = range(1, 31)
train_accuracies = []
test_accuracies = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    
    train_acc = accuracy_score(y_train, knn.predict(X_train_scaled))
    test_acc = accuracy_score(y_test, knn.predict(X_test_scaled))
    
    train_accuracies.append(train_acc)
    test_accuracies.append(test_acc)

# Best K dhundo
best_k = k_values[np.argmax(test_accuracies)]
best_acc = max(test_accuracies)

plt.figure(figsize=(12, 5))
plt.plot(k_values, train_accuracies, 'b-o', markersize=5, label='Training Accuracy', linewidth=2)
plt.plot(k_values, test_accuracies, 'r-s', markersize=5, label='Testing Accuracy', linewidth=2)
plt.axvline(x=best_k, color='green', linestyle='--', linewidth=2, label=f'Best K = {best_k}')
plt.scatter([best_k], [best_acc], color='gold', s=200, zorder=5, edgecolor='black')
plt.xlabel('K ki Value (Kitne Neighbours)', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Elbow Method — Sabse Achha K Dhundho!', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.xticks(range(1, 31))
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

print(f"\n🏆 Best K = {best_k}")
print(f"   Test Accuracy = {best_acc:.4f} ({best_acc*100:.2f}%)")

## Step 6: Model Training — KNN Model Banao
---
**Keyword Explanation:**
- **n_neighbors** — K ki value, kitne neighbors dekhne hain
- **metric** — kaunsi distance formula use karni hai
  - `'euclidean'` — seedhi line ki doori
  - `'manhattan'` — grid path ki doori
  - `'minkowski'` — generalized formula (default, p=2 means euclidean)
- **weights** — `'uniform'` (sab equal) ya `'distance'` (paas wale ka zyada effect)

In [ ]:
# Best K se model train karo
knn_model = KNeighborsClassifier(
    n_neighbors=best_k,
    metric='minkowski',  # Default distance (Euclidean jab p=2)
    p=2,                 # p=2 means Euclidean distance
    weights='uniform'    # Sabhi neighbors ka equal weight
)

knn_model.fit(X_train_scaled, y_train)
print(f"✅ KNN model train ho gaya! (K = {best_k})")

# Predictions
y_pred = knn_model.predict(X_test_scaled)

# Accuracy check
train_accuracy = accuracy_score(y_train, knn_model.predict(X_train_scaled))
test_accuracy = accuracy_score(y_test, y_pred)

print(f"\nModel Performance:")
print(f"  Training Accuracy:  {train_accuracy:.4f} ({train_accuracy*100:.2f}%)")
print(f"  Testing Accuracy:   {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

In [ ]:
# Alag-alag distance metrics compare karo
print("Alag-alag Distance Metrics ka Comparison:")
print("-" * 50)

metrics = [
    ('euclidean', 2, 'Euclidean (Seedhi Doori)'),
    ('manhattan', 1, 'Manhattan (Grid Path)'),
    ('minkowski', 3, 'Minkowski (p=3)')
]

for metric, p, name in metrics:
    if metric == 'manhattan':
        knn_temp = KNeighborsClassifier(n_neighbors=best_k, metric=metric)
    elif metric == 'minkowski':
        knn_temp = KNeighborsClassifier(n_neighbors=best_k, metric=metric, p=p)
    else:
        knn_temp = KNeighborsClassifier(n_neighbors=best_k, metric=metric)
    
    knn_temp.fit(X_train_scaled, y_train)
    acc = accuracy_score(y_test, knn_temp.predict(X_test_scaled))
    print(f"  {name}: {acc:.4f} ({acc*100:.2f}%)")

## Step 7: Confusion Matrix — Model ne Kya Galat Kiya?
---
**Confusion Matrix recap:**
- Rows = **Actual** (sach mein kya tha)
- Columns = **Predicted** (model ne kya bola)
- Diagonal = **Sahi predictions** ✅
- Off-diagonal = **Galat predictions** ❌

**Purana Tarika vs Naya Tarika:**
| Feature | Purana (Seaborn Heatmap) | Naya (ConfusionMatrixDisplay) |
|---------|--------------------------|--------------------------------|
| Import | seaborn | sklearn.metrics |
| Labels | Manual daalne padte hain | Auto-detect karta hai |
| Color | Custom | Built-in |
| Preferred | Traditional | Industry standard (2023+) |

In [ ]:
# Confusion Matrix banao
cm = confusion_matrix(y_test, y_pred)
species_names = ['Setosa', 'Versicolor', 'Virginica']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- PURANA TARIKA: Seaborn Heatmap ---
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=species_names, yticklabels=species_names,
            linewidths=0.5, linecolor='gray',
            annot_kws={'size': 14, 'weight': 'bold'},
            ax=axes[0])
axes[0].set_title('Purana Tarika: Seaborn Heatmap', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Predicted Label (Model ne kya bola)', fontsize=10)
axes[0].set_ylabel('Actual Label (Sach mein kya tha)', fontsize=10)

# --- NAYA TARIKA: ConfusionMatrixDisplay ---
cmd = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=species_names)
cmd.plot(cmap='Greens', colorbar=False, ax=axes[1])
axes[1].set_title('Naya Tarika: ConfusionMatrixDisplay', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Predicted Label (Model ne kya bola)', fontsize=10)
axes[1].set_ylabel('Actual Label (Sach mein kya tha)', fontsize=10)

plt.suptitle('Confusion Matrix — Do Tarikon Se!', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Confusion Matrix ko samjho
print("\nConfusion Matrix Explanation:")
for i, species in enumerate(species_names):
    correct = cm[i, i]
    total = cm[i].sum()
    print(f"  {species}: {correct}/{total} sahi predict kiye ({correct/total*100:.1f}%)")

## Step 8: Precision, Recall, F1 Score
---
**Keyword Explanation:**
- **Precision** — Jab model ne "Haan" bola, kitni baar sach tha?
- **Recall** — Jo actually positive tha, kitne pakad paaye?
- **F1 Score** — Precision aur Recall ka balance
- **macro avg** — Sabhi classes ka average (equal weight)
- **weighted avg** — Classes ki size ke hisaab se weighted average

In [ ]:
# Detailed Classification Report
print("=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=species_names))

In [ ]:
# Precision, Recall, F1 ka Bar Chart
from sklearn.metrics import precision_score, recall_score, f1_score

precision = precision_score(y_test, y_pred, average=None)
recall = recall_score(y_test, y_pred, average=None)
f1 = f1_score(y_test, y_pred, average=None)

x = np.arange(len(species_names))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 6))

bars1 = ax.bar(x - width, precision, width, label='Precision', color='#FF6B6B', edgecolor='black')
bars2 = ax.bar(x, recall, width, label='Recall', color='#4ECDC4', edgecolor='black')
bars3 = ax.bar(x + width, f1, width, label='F1 Score', color='#45B7D1', edgecolor='black')

ax.set_xlabel('Species (Phool ka Prakar)', fontsize=12)
ax.set_ylabel('Score (0 se 1)', fontsize=12)
ax.set_title('Precision, Recall, F1 Score — Har Species ke Liye', 
             fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(species_names)
ax.set_ylim(0, 1.15)
ax.legend(fontsize=11)
ax.grid(True, axis='y', alpha=0.4)

# Values dikhao bars ke upar
for bar_group in [bars1, bars2, bars3]:
    for bar in bar_group:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{height:.2f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## Step 9: Decision Boundary — Model Kaise Sochta Hai?
---
**Decision Boundary kya hota hai?**
> Wo imaginary line jo model ne draw ki hai different classes ke beech.
> KNN ki boundary jagged (ulti-pulti) hoti hai, smooth nahi.

**Note:** Sirf 2 features use karenge visualization ke liye (petal_length aur petal_width)
kyunki 2D mein hi plot kar sakte hain.

In [ ]:
# Decision Boundary — Sirf 2 Features (petal_length, petal_width)
X_2d = df[['petal_length', 'petal_width']].values
y_2d = df['species_encoded'].values

X_train_2d, X_test_2d, y_train_2d, y_test_2d = train_test_split(
    X_2d, y_2d, test_size=0.2, random_state=42, stratify=y_2d
)

scaler_2d = StandardScaler()
X_train_2d_s = scaler_2d.fit_transform(X_train_2d)
X_test_2d_s = scaler_2d.transform(X_test_2d)

# K=1, K=5, K=15 ke boundaries compare karo
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
k_to_show = [1, 5, 15]
colors_map = ['#FF6B6B', '#4ECDC4', '#45B7D1']

for ax, k in zip(axes, k_to_show):
    knn_2d = KNeighborsClassifier(n_neighbors=k)
    knn_2d.fit(X_train_2d_s, y_train_2d)
    
    # Mesh grid banao
    x_min, x_max = X_train_2d_s[:, 0].min() - 0.5, X_train_2d_s[:, 0].max() + 0.5
    y_min, y_max = X_train_2d_s[:, 1].min() - 0.5, X_train_2d_s[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                          np.arange(y_min, y_max, 0.02))
    
    Z = knn_2d.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
    for class_idx, color in enumerate(colors_map):
        mask = y_train_2d == class_idx
        ax.scatter(X_train_2d_s[mask, 0], X_train_2d_s[mask, 1],
                   c=color, label=species_names[class_idx], 
                   edgecolor='black', s=50, alpha=0.8)
    
    acc = accuracy_score(y_test_2d, knn_2d.predict(X_test_2d_s))
    ax.set_title(f'K = {k}\nAccuracy = {acc:.2f}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Petal Length (scaled)')
    ax.set_ylabel('Petal Width (scaled)')
    ax.legend(fontsize=8)

plt.suptitle('Decision Boundary — K ki Value ka Asar', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print("Dekho: K=1 → Jagged boundary (overfitting), K=15 → Smooth boundary (better generalization)")

## Step 10: Final Performance Summary
---

In [ ]:
# Final Summary
print("=" * 60)
print("FINAL PERFORMANCE SUMMARY")
print("=" * 60)
print(f"\nDataset: Iris Flowers (3 species)")
print(f"Total Samples: {len(df)}")
print(f"Training Samples: {len(y_train)} | Test Samples: {len(y_test)}")
print(f"Best K: {best_k}")
print(f"Distance Metric: Minkowski (p=2 = Euclidean)")
print(f"Scaling: StandardScaler")
print()
print(f"Training Accuracy: {accuracy_score(y_train, knn_model.predict(X_train_scaled))*100:.2f}%")
print(f"Testing Accuracy:  {test_accuracy*100:.2f}%")
print()
print("Per-Class Performance:")
print(f"{'Species':<15} {'Precision':<12} {'Recall':<12} {'F1 Score':<12}")
print("-" * 51)
for i, species in enumerate(species_names):
    print(f"{species:<15} {precision[i]:<12.4f} {recall[i]:<12.4f} {f1[i]:<12.4f}")
print("=" * 60)

In [ ]:
# Naya sample predict karo!
print("Naya Flower Classify Karo!")
print("=" * 40)

# Ek nayi flower ki measurements do
new_flower = np.array([[5.1, 3.5, 1.4, 0.2]])  # Likely Setosa

new_flower_scaled = scaler.transform(new_flower)
prediction = knn_model.predict(new_flower_scaled)
probabilities = knn_model.predict_proba(new_flower_scaled)

print(f"Input Measurements:")
for feat, val in zip(features, new_flower[0]):
    print(f"  {feat}: {val}")
print(f"\nPrediction: {le.inverse_transform(prediction)[0].upper()}")
print("\nProbabilities:")
for species, prob in zip(species_names, probabilities[0]):
    bar = '█' * int(prob * 20)
    print(f"  {species:<12}: {bar:<20} {prob*100:.1f}%")

## KNN Capstone Project 🎯
---
**Tumhara Assignment:**

Iris dataset par ek complete KNN pipeline banao with ye changes:

**Part A — Exploration:**
1. Sirf 2 features (sepal_length, sepal_width) use karke model banao
2. Accuracy compare karo full 4 features vs 2 features
3. KNN with `weights='distance'` try karo — kya accuracy improve hoti hai?

**Part B — Distance Comparison:**
1. Euclidean, Manhattan, aur Minkowski (p=4) metrics compare karo ek table mein
2. Har metric ke liye best K bhi dhundo

**Part C — Advanced:**
1. 5-Fold Cross Validation use karo (hint: `from sklearn.model_selection import cross_val_score`)
2. Report karo: Mean Accuracy ± Standard Deviation

**Bonus Challenge:**
- KNN Regression use karo: `sepal_length` predict karo baaki features se
- Use `KNeighborsRegressor` aur `mean_squared_error` report karo

In [ ]:
# Part A — Yahan apna code likho
print("Part A: Sirf 2 Features se Model")
print("Yahan Apna Code Likho! 🚀")

# Hint:
# X_2feat = df[['sepal_length', 'sepal_width']].values
# y_2feat = df['species_encoded'].values
# ... (baaki pipeline same rahegi)

In [ ]:
# Part B — Distance Metrics Comparison Table
print("Part B: Distance Metrics Comparison")
print("Yahan Apna Code Likho! 📊")

# Hint:
# results = []
# for metric in ['euclidean', 'manhattan']:
#     for k in range(1, 21):
#         knn = KNeighborsClassifier(n_neighbors=k, metric=metric)
#         knn.fit(X_train_scaled, y_train)
#         acc = accuracy_score(y_test, knn.predict(X_test_scaled))
#         results.append({'K': k, 'Metric': metric, 'Accuracy': acc})
# pd.DataFrame(results).pivot(index='K', columns='Metric', values='Accuracy')

In [ ]:
# Part C — Cross Validation
print("Part C: 5-Fold Cross Validation")
print("Yahan Apna Code Likho! 🔄")

# Hint:
# from sklearn.model_selection import cross_val_score
# knn = KNeighborsClassifier(n_neighbors=best_k)
# scores = cross_val_score(knn, X, y, cv=5, scoring='accuracy')
# print(f"Cross-Val Mean: {scores.mean():.4f} ± {scores.std():.4f}")

---
## Notebook Complete! 🎉

**Aaj humne sikha:**

| Topic | Kya Sikha |
|-------|----------|
| Data Loading | `sns.load_dataset('iris')` se real data load karna |
| EDA | Pairplot, Boxplot, Heatmap se patterns samajhna |
| Preprocessing | LabelEncoder, StandardScaler, Train-Test Split |
| Elbow Method | Best K dhundhna — overfitting vs underfitting |
| KNN Model | `KNeighborsClassifier` fit karna aur predict karna |
| Distance Metrics | Euclidean, Manhattan, Minkowski compare karna |
| Evaluation | Confusion Matrix (dono methods), Precision/Recall/F1 |
| Decision Boundary | Model ka thinking process visualize karna |

**Next Step:** KNN Capstone Project complete karo! 💪

---
*Atlas Skilltech University — Machine Learning Course*